# Treino do YOLOv8n custom (variante CholecTrack20)

Variante do treino YOLO usando **CholecTrack20** (Synapse syn53182642) em
vez de CholecSeg8k. CholecTrack20 oferece:

- 35k frames anotados (vs 8k de CholecSeg8k)
- 7 classes de instrumento (vs 2 do treino baseline em CholecSeg8k)
- Bounding boxes nativos (vs conversao de mascara semantica)
- Splits oficiais 10/2/8

Pre-requisito: acesso aprovado ao dataset no Synapse + Personal Access Token (PAT).
Solicitar acesso em https://www.synapse.org/Synapse:syn53182642/wiki/628404

Output equivalente ao notebook baseline: pesos finais em
`MyDrive/medica-ia/yolo_runs_cholectrack20/surgical_instruments_ct20/weights/best.pt`

## 1. Setup

Esta secao prepara o ambiente em 3 partes:

**1.1 GPU + batch:** detecta automaticamente o GPU disponivel via `nvidia-smi` e seleciona o batch/workers apropriado. Suporta T4 (gratuita, 15 GB VRAM), L4 (Colab Pro, 22-24 GB, ~2.5x mais rapido) e A100 (Colab Pro+, 40 GB, ~5x mais rapido).

| GPU | VRAM | Batch padrao | Workers | Velocidade relativa |
|-----|------|--------------|---------|---------------------|
| T4 | 15 GB | 96 | 8 | 1x (baseline) |
| L4 | 22-24 GB | 160 | 12 | ~2.5-3x |
| A100 | 40 GB | 320 | 16 | ~5x |

Para mudar a GPU: `Runtime > Change runtime type > GPU type`. A celula detecta a nova GPU automaticamente.

**1.2 Drive:** usado apenas para artefatos pequenos e criticos que valem persistir entre sessoes:
- `yolo_runs_cholectrack20/<run_name>/weights/`: `best.pt` e `last.pt` (~6-12 MB cada)
- `yolo_runs_cholectrack20/<run_name>/`: graficos (`results.png`, `confusion_matrix.png`), CSV (`results.csv`)

**Dataset (raw ~XX GB + convertido) NAO e cacheado no Drive** porque ler do Drive provou-se mais lento do que re-baixar diretamente. Em cada nova sessao do Colab o dataset e baixado e convertido localmente em `/content/`.

Em re-execucao com a mesma sessao do Colab ativa, o local (`/content/`) ja tem o dataset e o treino pula direto pro fine-tuning. Se o Colab desconectar, o `last.pt` salvo no Drive permite resumir o treino (callback `on_train_epoch_end` sincroniza pesos por epoca).

**1.3 Synapse auth:** CholecTrack20 exige DUA aprovado + PAT para download. Preencher `SYNAPSE_TOKEN` antes de prosseguir.

In [ ]:
# 1.1 GPU + batch (auto-detect via nvidia-smi)
# A escolha do tipo de GPU e manual no Colab: Runtime > Change runtime type.
# Esta celula apenas detecta qual GPU foi alocada e ajusta batch/workers.
import subprocess

GPU_CONFIGS = {
    'T4':  {'batch': 96,  'workers': 8,  'vram_gb': 15, 'note': 'gratuito'},
    'L4':  {'batch': 160, 'workers': 12, 'vram_gb': 22, 'note': 'Colab Pro, ~2.5x mais rapido que T4'},
    'A100':{'batch': 320, 'workers': 16, 'vram_gb': 40, 'note': 'Colab Pro+, ~5x mais rapido que T4'},
}

def _detect_gpu():
    """Roda nvidia-smi e retorna a chave de GPU_CONFIGS, ou None."""
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True, timeout=10,
        ).strip()
    except Exception as exc:
        print(f'[gpu] Falha ao rodar nvidia-smi: {exc}')
        return None, None

    first = out.split('\n')[0]
    upper = first.upper()
    for key in ('A100', 'L4', 'T4'):
        if key in upper:
            return key, first
    return None, first

# Override manual: descomente pra forcar (ex: testar limites)
# GPU_TYPE = 'T4'

_detected, _raw_name = _detect_gpu()
if 'GPU_TYPE' not in dir() or GPU_TYPE is None:
    GPU_TYPE = _detected or 'T4'

cfg = GPU_CONFIGS[GPU_TYPE]
BATCH_SIZE = cfg['batch']
WORKERS    = cfg['workers']

print(f'[gpu] nvidia-smi reportou: {_raw_name!r}')
print(f'[gpu] Mapeado para: {GPU_TYPE} ({cfg["vram_gb"]} GB VRAM, {cfg["note"]})')
print(f'[gpu] Batch size: {BATCH_SIZE}, workers: {WORKERS}')
if _detected is None and _raw_name:
    print(f'[gpu] AVISO: GPU nao mapeada explicitamente em GPU_CONFIGS, usando fallback T4.')
    print(f'[gpu] Pra usar mais VRAM, descomentar `GPU_TYPE = ...` acima.')

print()
!nvidia-smi

In [ ]:
# 1.2 Drive (so para artefatos pequenos do treino: best.pt, last.pt, graficos)
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

DRIVE_ROOT  = Path('/content/drive/MyDrive/medica-ia')
DRIVE_RUNS  = DRIVE_ROOT / 'yolo_runs_cholectrack20'
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

# Paths locais no Colab (dataset bruto e convertido NAO sincronizam pro Drive:
# re-baixar do Synapse e mais rapido do que ler do Drive por sessao)
DATASET_DIR = '/content/cholectrack20_yolo'
RAW_DIR     = '/content/cholectrack20_raw'
RUNS_DIR    = '/content/runs/detect'
RUN_NAME    = 'surgical_instruments_ct20'
DATA_YAML   = f'{DATASET_DIR}/data.yaml'

print()
print(f'Drive root:           {DRIVE_ROOT}')
print(f'  runs (output):      {DRIVE_RUNS}')
print(f'Working dir (Colab):  {DATASET_DIR} (efemero, nao sincroniza pro Drive)')

In [ ]:
# 1.3 Synapse auth + accesskey CholecTrack20
# Para baixar CholecTrack20 voce precisa de 3 coisas:
# 1. Conta Synapse (com email) - https://www.synapse.org
# 2. Personal Access Token (PAT) com permissao "Download"
#    https://www.synapse.org/Profile:v/settings (Account > Personal Access Tokens)
# 3. accesskey enviada por email apos aprovacao do form
#    https://docs.google.com/forms/d/e/1FAIpQLSdewhAi0vGmZj5DLOMWdLf85BhUtTedS28YzvHS58ViwuEX5w/viewform

SYNAPSE_EMAIL       = ""  # TODO: email cadastrado no Synapse
SYNAPSE_TOKEN       = ""  # TODO: Personal Access Token (autoToken)
CHOLECTRACK_ACCESSKEY = ""  # TODO: accesskey recebida por email apos aprovacao

assert SYNAPSE_EMAIL,       "Preencher SYNAPSE_EMAIL"
assert SYNAPSE_TOKEN,       "Preencher SYNAPSE_TOKEN"
assert CHOLECTRACK_ACCESSKEY, "Preencher CHOLECTRACK_ACCESSKEY (vem por email apos aprovar form CAMMA)"

!pip install -q synapseclient ultralytics==8.3.30 requests
import synapseclient
syn = synapseclient.login(email=SYNAPSE_EMAIL, authToken=SYNAPSE_TOKEN, silent=True)

# Armazena perfil pra reuso (API nova, sem deprecation warning)
from synapseclient.models import UserProfile
_me = UserProfile().get()
SYNAPSE_USER_ID = _me.id
print(f"Logado como: {_me.username} (id={SYNAPSE_USER_ID})")


## 2. Dataset

Pipeline em 5 passos: verifica cache + baixa do Synapse, extrai arquivos, converte anotacoes para formato YOLO, gera `data.yaml`, valida com visualizacao. Tudo idempotente.

### 2.1 Cache + download

Verifica se o dataset convertido ja esta no working dir do Colab. Se nao, baixa do Synapse via `syn.get()`. O dataset raw NAO e cacheado no Drive (re-baixar e mais rapido que ler do Drive).

CholecTrack20 tem 20 videos de colecistectomia laparoscopica com ~35k frames @ 1 fps, 65k labels de instancia. Splits oficiais: 10 videos train / 2 val / 8 test.

In [ ]:
from pathlib import Path

def split_is_complete(base_dir: str, split: str, min_count: int) -> bool:
    """Conta arquivos em <base_dir>/images/<split>, tolerante a erros de FS."""
    img_dir = Path(base_dir) / 'images' / split
    try:
        if not img_dir.is_dir():
            return False
        n = sum(1 for _ in img_dir.iterdir())
        return n >= min_count
    except OSError as exc:
        print(f'[cache] aviso: erro ao ler {img_dir}: {exc}')
        return False

# Contagens minimas esperadas por split (splits oficiais CholecTrack20: 10/2/8 videos)
EXPECTED = {'train': 15000, 'val': 3000, 'test': 12000}

local_ready = all(split_is_complete(DATASET_DIR, s, n) for s, n in EXPECTED.items())

if local_ready:
    NEEDS_PREPARE = False
    print(f'[cache] OK: dataset ja no Colab em {DATASET_DIR}')
else:
    NEEDS_PREPARE = True
    print('[cache] Dataset nao encontrado no Colab. Iniciando download do Synapse...')

    raw_path = Path(RAW_DIR)
    raw_path.mkdir(parents=True, exist_ok=True)

    # Fluxo oficial CholecTrack20 (descoberto em syn53182642/wiki/628404):
    # 1. Login Synapse (ja feito na celula 1.3)
    # 2. Validar accesskey via API custom da CAMMA
    #    -> retorna entity_id dinamico autorizado pra sua conta
    # 3. syncFromSynapse(entity_id) baixa hierarquia completa
    #
    # Por que syn53182642 nao funciona direto: o projeto raiz nao da
    # acesso aos arquivos. A CAMMA tem um proxy de autorizacao que devolve
    # o entity_id correto so apos validar a accesskey.

    import requests
    import synapseutils

    print('[auth] Validando accesskey via API da CAMMA...')
    CAMMA_API_URL = "https://synapse-response.onrender.com/validate_access"
    USER_ID = SYNAPSE_USER_ID  # definido na celula 1.3

    response = requests.post(
        CAMMA_API_URL,
        json={"access_key": CHOLECTRACK_ACCESSKEY, "synapse_id": USER_ID},
    )

    if response.status_code != 200:
        raise RuntimeError(
            f'[auth] Falha ao validar accesskey ({response.status_code}):\n'
            f'{response.text}\n\n'
            'Possiveis causas:\n'
            '  - accesskey errada ou expirada\n'
            '  - USER_ID errado (Synapse account diferente do aprovado)\n'
            '  - servico CAMMA fora do ar (tenta de novo em alguns minutos)'
        )

    entity_id = response.json().get('entity_id')
    if not entity_id:
        raise RuntimeError(f'[auth] API retornou 200 mas sem entity_id: {response.json()}')

    print(f'[auth] OK: entity autorizado = {entity_id}')

    # Download hierarquico
    print(f'\n[download] Baixando arvore inteira de {entity_id} para {RAW_DIR}...')
    synced_files = synapseutils.syncFromSynapse(syn, entity=entity_id, path=RAW_DIR)
    print(f'[download] {len(synced_files)} arquivo(s) baixado(s).')

    print(f'\n[diag] Conteudo de RAW_DIR apos download:')
    !ls -lh {RAW_DIR}/ | head -20


### 2.2 Extracao

Descompacta arquivos baixados do Synapse em `RAW_DIR/extracted/`. Usa `zipfile` + `tqdm` para mostrar progresso. Se o dataset ja vier descompactado (pasta com JSONs/imagens), esta celula detecta e pula a extracao.

In [ ]:
if NEEDS_PREPARE:
    import zipfile
    from pathlib import Path
    from tqdm.auto import tqdm

    raw_path      = Path(RAW_DIR)
    extracted_root = raw_path / 'extracted'

    # syncFromSynapse pode trazer arquivos ja descompactados (estrutura de pastas)
    # OU zips por video. Detectamos as duas situacoes.

    # Verifica se ja existe conteudo extraido
    already_extracted = extracted_root.exists() and any(extracted_root.iterdir())

    if already_extracted:
        print(f'Conteudo ja extraido em {extracted_root}, pulando.')
    else:
        # Procura zips recursivamente em RAW_DIR (synapseutils preserva subpastas)
        zips = list(raw_path.rglob('*.zip'))
        if not zips:
            # Dataset veio como hierarquia de pastas/arquivos (caso comum do CholecTrack20)
            print('[info] Nenhum .zip encontrado. Dataset provavelmente veio como pasta direta.')
            print(f'[info] Estrutura de {raw_path}:')
            !find {raw_path} -maxdepth 3 -type d | head -30
            extracted_root = raw_path  # processa direto da raiz baixada
        else:
            extracted_root.mkdir(parents=True, exist_ok=True)
            print(f'Encontrados {len(zips)} arquivo(s) zip.')
            for zp in zips:
                print(f'Extraindo {zp.name} -> {extracted_root}')
                with zipfile.ZipFile(zp) as zf:
                    members = zf.namelist()
                    for member in tqdm(members, desc=f'Extraindo {zp.name}', unit='arq'):
                        zf.extract(member, extracted_root)
                print(f'  {len(members)} arquivos extraidos.')

    EXTRACTED_ROOT = extracted_root
    print(f'\nRaiz dos dados brutos: {EXTRACTED_ROOT}')
    print(f'Amostra de JSONs encontrados (primeiros 5):')
    sample_jsons = list(EXTRACTED_ROOT.rglob('*.json'))[:5]
    for j in sample_jsons:
        print(f'  {j.relative_to(EXTRACTED_ROOT)}')
    print(f'Total de JSONs: {len(list(EXTRACTED_ROOT.rglob("*.json")))}')
else:
    print('Pulado (cache).')


### 2.3 Conversao para formato YOLO

CholecTrack20 vem com **um JSON por video** (`VIDxx/VIDxx.json`), keyed por frame_id (string), com lista de instancias por frame. Cada instancia tem:

- `instrument`: class id (0-6)
- `tool_bbox`: `[x, y, w, h]` ja normalizado (top-left xywh)
- `visible`: 0/1 (instancias com `visible=0` sao descartadas)
- Flags de visual challenge: `bleeding`, `smoke`, `occluded`, `blurred`, etc. (booleans por instancia)

A conversao:

1. Itera Training/Validation/Testing -> mapeia pra splits YOLO train/val/test
2. Para cada `VIDxx/VIDxx.json`, processa cada frame e cada instancia visivel
3. Converte `tool_bbox` (top-left xywh normalizado) -> center xywh normalizado (formato YOLO)
4. Escreve label `.txt` em `labels/<split>/<VID>_<frame_id>.txt`
5. Cria **symlink** da PNG (em vez de copia) em `images/<split>/<VID>_<frame_id>.png` (poupa ~28 GB)
6. Salva manifest `bleeding_frames.txt` com IDs de frames flagged como `bleeding=1` (consumido depois pela conversao v3 combinada)

**Decisao sobre a classe `blood`:**
Na primeira iteracao (v2), apenas os 7 instrumentos sao treinados. A flag `bleeding` por instancia e armazenada no manifest mas nao vira classe propria, ja que nao tem bbox dedicada (e atributo da instancia de instrumento). A inclusao de uma classe explicita de sangramento e avaliada na etapa de combinacao com CholecSeg8k.

**7 classes de instrumento mapeadas:**

| ID | Classe |
|----|--------|
| 0 | grasper |
| 1 | bipolar |
| 2 | hook |
| 3 | scissors |
| 4 | clipper |
| 5 | irrigator |
| 6 | specimen_bag |

In [ ]:
if NEEDS_PREPARE:
    import json
    import shutil
    from pathlib import Path
    from tqdm.auto import tqdm

    INSTRUMENT_NAMES = ['grasper', 'bipolar', 'hook', 'scissors',
                        'clipper', 'irrigator', 'specimen_bag']

    # Pasta no dataset raw -> nome do split YOLO
    SPLIT_MAP = {'Training': 'train', 'Validation': 'val', 'Testing': 'test'}

    raw_root  = Path(RAW_DIR)
    yolo_root = Path(DATASET_DIR)

    # Conversao idempotente: limpa saida anterior
    if yolo_root.exists():
        shutil.rmtree(yolo_root)
    for split in ('train', 'val', 'test'):
        (yolo_root / 'images' / split).mkdir(parents=True, exist_ok=True)
        (yolo_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

    stats = {'frames': 0, 'anns': 0, 'skipped_no_visible': 0,
             'bleeding_frames': 0, 'missing_png': 0}
    # Frames com bleeding flag (consumido depois pela conversao v3 combinada)
    bleeding_manifest = []

    for raw_split, yolo_split in SPLIT_MAP.items():
        split_dir = raw_root / raw_split
        if not split_dir.exists():
            print(f'WARN: split dir nao existe: {split_dir}')
            continue

        vid_dirs = sorted(d for d in split_dir.iterdir()
                          if d.is_dir() and d.name.startswith('VID'))
        print(f'\n[{raw_split} -> {yolo_split}] {len(vid_dirs)} videos')

        for vid_dir in vid_dirs:
            json_path  = vid_dir / f'{vid_dir.name}.json'
            frames_dir = vid_dir / 'Frames'
            if not json_path.exists() or not frames_dir.exists():
                print(f'  SKIP {vid_dir.name}: faltando JSON ou Frames/')
                continue

            # CT20 schema: {info, annotations, categories, video}
            # As anotacoes por frame ficam em anns_data['annotations']
            anns_data = json.loads(json_path.read_text())
            anns = anns_data.get('annotations', {})
            if not isinstance(anns, dict):
                print(f'  SKIP {vid_dir.name}: campo annotations nao e dict')
                continue

            n_frames_vid = 0
            for frame_id_str, instances in anns.items():
                # Pula chaves nao-numericas (defensivo)
                if not frame_id_str.isdigit():
                    continue
                if not isinstance(instances, list):
                    continue
                frame_id = int(frame_id_str)
                png_name = f'{frame_id:06d}.png'
                png_path = frames_dir / png_name
                if not png_path.exists():
                    stats['missing_png'] += 1
                    continue

                lines = []
                has_bleeding = False
                for inst in instances:
                    if inst.get('visible', 1) == 0:
                        continue
                    cls_id = inst.get('instrument', -1)
                    if cls_id < 0 or cls_id >= len(INSTRUMENT_NAMES):
                        continue
                    x, y, w, h = inst['tool_bbox']
                    # tool_bbox vem como top-left xywh normalizado;
                    # YOLO espera center xywh normalizado
                    cx = max(0.0, min(1.0, x + w / 2))
                    cy = max(0.0, min(1.0, y + h / 2))
                    w  = max(0.0, min(1.0, w))
                    h  = max(0.0, min(1.0, h))
                    lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
                    if inst.get('bleeding', 0) == 1:
                        has_bleeding = True

                if not lines:
                    stats['skipped_no_visible'] += 1
                    continue

                # Nome unico por split (sem colisao entre videos): <VID>_<frame_id>
                out_stem = f'{vid_dir.name}_{frame_id:06d}'
                img_link = yolo_root / 'images' / yolo_split / f'{out_stem}.png'
                lbl_file = yolo_root / 'labels' / yolo_split / f'{out_stem}.txt'

                # Symlink ao inves de copia: poupa ~28 GB de disco
                if not img_link.exists():
                    img_link.symlink_to(png_path.resolve())
                lbl_file.write_text('\n'.join(lines))

                stats['frames'] += 1
                stats['anns']  += len(lines)
                n_frames_vid   += 1
                if has_bleeding:
                    stats['bleeding_frames'] += 1
                    bleeding_manifest.append(f'{yolo_split}/{out_stem}')

            print(f'  {vid_dir.name}: {n_frames_vid} frames')

    (yolo_root / 'bleeding_frames.txt').write_text('\n'.join(bleeding_manifest))

    print('\n=== Resumo da conversao ===')
    print(f'Total frames convertidos:    {stats["frames"]}')
    print(f'Total annotations:           {stats["anns"]}')
    print(f'Frames com bleeding flag:    {stats["bleeding_frames"]}')
    print(f'Pulados (sem inst visivel):  {stats["skipped_no_visible"]}')
    print(f'PNGs missing (ref JSON):     {stats["missing_png"]}')
else:
    print(f'Reuso da conversao anterior em {DATASET_DIR}')

### 2.4 data.yaml

Aponta para a raiz do dataset no Colab com as 7 classes de instrumento.

In [ ]:
import yaml

with open(DATA_YAML, 'w') as f:
    yaml.safe_dump({
        'path':  DATASET_DIR,
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'nc':    7,
        'names': {
            0: 'grasper',
            1: 'bipolar',
            2: 'hook',
            3: 'scissors',
            4: 'clipper',
            5: 'irrigator',
            6: 'specimen_bag',
        },
    }, f, sort_keys=False)

print(open(DATA_YAML).read())

### 2.5 Validacao + visualizacao

Sanity check: contagem por split e quatro amostras com bbox sobre o frame.

In [ ]:
import os

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    n_imgs     = len(os.listdir(img_dir))
    n_nonempty = sum(1 for f in os.listdir(lbl_dir)
                     if os.path.getsize(os.path.join(lbl_dir, f)) > 0)
    print(f'{split:>5}: {n_imgs:>6} imagens ({n_nonempty} com bbox)')

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(42)

# Paleta de cores por classe (7 classes)
CLASS_COLORS = {
    0: 'lime',
    1: 'cyan',
    2: 'magenta',
    3: 'yellow',
    4: 'orange',
    5: 'deepskyblue',
    6: 'tomato',
}
CLASS_LABELS = {
    0: 'grasper',
    1: 'bipolar',
    2: 'hook',
    3: 'scissors',
    4: 'clipper',
    5: 'irrigator',
    6: 'specimen_bag',
}

img_dir = os.path.join(DATASET_DIR, 'images', 'train')
lbl_dir = os.path.join(DATASET_DIR, 'labels', 'train')

candidates = [
    f for f in os.listdir(img_dir)
    if os.path.getsize(os.path.join(lbl_dir, f.replace('.png', '.txt'))) > 0
]

if not candidates:
    raise RuntimeError(
        f'Nenhuma imagem com bbox em {lbl_dir}. '
        'A conversao da 2.3 nao gerou labels (ou o cache em DATASET_DIR ficou inconsistente). '
        'Apague DATASET_DIR e rode tudo de novo a partir da 2.1.'
    )

samples = random.sample(candidates, min(4, len(candidates)))
n = len(samples)
rows = (n + 1) // 2
fig, axes = plt.subplots(rows, 2, figsize=(14, 5 * rows))
axes_list = list(axes.flat) if n > 1 else [axes]

for ax, name in zip(axes_list, samples):
    img = Image.open(os.path.join(img_dir, name))
    W, H = img.size
    ax.imshow(img)
    ax.set_title(name, fontsize=9)
    ax.axis('off')

    lbl_name = name.replace('.png', '.txt')
    with open(os.path.join(lbl_dir, lbl_name)) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            xc, yc, w, h = map(float, parts[1:5])
            x0 = (xc - w / 2) * W
            y0 = (yc - h / 2) * H
            ax.add_patch(patches.Rectangle(
                (x0, y0), w * W, h * H,
                linewidth=2,
                edgecolor=CLASS_COLORS.get(cls, 'white'),
                facecolor='none',
            ))
            ax.text(
                x0, max(0, y0 - 5),
                CLASS_LABELS.get(cls, str(cls)),
                color=CLASS_COLORS.get(cls, 'white'),
                fontsize=9,
                bbox=dict(facecolor='black', alpha=0.6, pad=2),
            )

for ax in axes_list[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Treino

Treino na GPU configurada na celula 1.1 (`device=0` no `model.train()`). Partimos dos pesos `yolov8n.pt` (pre-treinado em COCO) e fazemos fine-tuning nas **7 classes alvo** de instrumento cirurgico. Escolhemos a variante **nano** porque o app Gradio do projeto roda os pesos finais em CPU local (sem GPU dedicada), entao a inferencia precisa ser leve. O treino em si nao tem essa restricao: usa a GPU do Colab.

`BATCH_SIZE` e `WORKERS` vem das constantes definidas em 1.1 (96/8 para T4, 160/12 para L4). Os valores foram calibrados para usar ~70-80% da VRAM disponivel sem risco de OOM.

**Resiliencia a desconexao do Colab:**
- Callback `on_train_epoch_end` copia `last.pt` e `best.pt` pro Drive ao fim de cada epoch
- Se a sessao cair, ao re-abrir o notebook o codigo detecta o checkpoint no Drive e oferece resumir via `resume=True`
- Sync final completo (graficos, csv, weights) ao terminar o treino

In [ ]:
import shutil
from pathlib import Path
from ultralytics import YOLO

# Detectar checkpoint anterior no Drive (caso sessao tenha caido)
drive_last    = DRIVE_RUNS / RUN_NAME / 'weights' / 'last.pt'
local_run_dir = Path(RUNS_DIR) / RUN_NAME

resume_flag = False
if drive_last.exists() and not (local_run_dir / 'weights' / 'last.pt').exists():
    size_mb = drive_last.stat().st_size / 1e6
    print(f'Checkpoint anterior no Drive ({size_mb:.1f} MB): {drive_last}')
    print('Copiando run dir do Drive para Colab e resumindo...')
    shutil.copytree(DRIVE_RUNS / RUN_NAME, local_run_dir, dirs_exist_ok=True)
    model = YOLO(str(local_run_dir / 'weights' / 'last.pt'))
    resume_flag = True
else:
    model = YOLO('yolov8n.pt')

# Callback: ao fim de cada epoch, copiar weights/last.pt e weights/best.pt pro Drive
def _sync_weights_to_drive(trainer) -> None:
    save_dir    = Path(trainer.save_dir)
    dst_weights = DRIVE_RUNS / RUN_NAME / 'weights'
    dst_weights.mkdir(parents=True, exist_ok=True)
    for wname in ('last.pt', 'best.pt'):
        src = save_dir / 'weights' / wname
        if src.exists():
            shutil.copy(src, dst_weights / wname)

model.add_callback('on_train_epoch_end', _sync_weights_to_drive)

# Treinar (ou resumir)
if resume_flag:
    results = model.train(resume=True)
else:
    results = model.train(
        data=DATA_YAML,
        epochs=40,
        imgsz=640,
        batch=BATCH_SIZE,
        workers=WORKERS,
        device=0,
        name=RUN_NAME,
        patience=10,
        project=RUNS_DIR,
        exist_ok=True,
        verbose=True,
    )

# Sync final completo (graficos, csv, etc) pro Drive
print('\nSincronizando run dir completo com o Drive...')
dst = DRIVE_RUNS / RUN_NAME
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(local_run_dir, dst)
print(f'OK: {dst}')

**Esperado:** `mAP50 > 0.7` ao fim do treino com 7 classes. CholecTrack20 tem mais frames e bboxes nativas (sem conversao de mascara), o que tende a produzir modelos com recall mais alto que o baseline CholecSeg8k. Se travar abaixo de 0.3 nas primeiras 5 epocas, revisar splits/labels.

## 4. Avaliacao

Metricas no split de teste + curvas + matriz de confusao + predicoes visuais.

In [ ]:
best_path  = f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt'
best_model = YOLO(best_path)

metrics = best_model.val(
    data=DATA_YAML,
    split='test',
    project=RUNS_DIR,
    name=f'{RUN_NAME}_test',
    exist_ok=True,
)

print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print()
print('mAP50 por classe:')
for i, name in metrics.names.items():
    print(f'  {name:<20s} {metrics.box.maps[i]:.4f}')

In [ ]:
from IPython.display import Image as IPImage, display

display(IPImage(f'{RUNS_DIR}/{RUN_NAME}/results.png'))
display(IPImage(f'{RUNS_DIR}/{RUN_NAME}_test/confusion_matrix.png'))

In [ ]:
import random
import matplotlib.pyplot as plt
import os

random.seed(42)
test_img_dir = os.path.join(DATASET_DIR, 'images', 'test')
test_samples = random.sample(os.listdir(test_img_dir), 4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, name in zip(axes.flat, test_samples):
    img_path = os.path.join(test_img_dir, name)
    pred     = best_model(img_path, conf=0.25, verbose=False)[0]
    annotated = pred.plot()
    ax.imshow(annotated[..., ::-1])
    ax.set_title(name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

### 4.5 Validacao visual: video anotado com instrumentos detectados

Cura uma sequencia de frames do **test split** que contenham o maior numero de instrumentos distintos, roda o `best.pt` em cada frame, anota com bboxes detectadas e empacota em MP4. Esse video serve como evidencia visual no relatorio tecnico de que o YOLO custom efetivamente detecta instrumentos cirurgicos em frames de colecistectomia laparoscopica.

O MP4 vai pro Drive em `yolo_runs_cholectrack20/<RUN_NAME>/validation_instruments.mp4`.

Nota: como a classe `blood` foi pulada nesta iteracao v2, a sequencia e escolhida com base na diversidade de instrumentos (mais classes distintas por frame).

In [ ]:
import re
import subprocess
import tempfile
import shutil as _shutil
from collections import defaultdict
from pathlib import Path
import cv2
import os

test_lbl_dir = Path(DATASET_DIR) / 'labels' / 'test'
test_img_dir = Path(DATASET_DIR) / 'images' / 'test'

# Contar classes distintas por frame para selecionar sequencia mais rica
frame_diversity: dict[str, int] = {}
for lbl_file in sorted(test_lbl_dir.iterdir()):
    if not lbl_file.is_file():
        continue
    content = lbl_file.read_text().strip()
    if not content:
        continue
    classes_present = set(line.split()[0] for line in content.split('\n') if line.strip())
    frame_diversity[lbl_file.stem + '.png'] = len(classes_present)

print(f'Frames de test com pelo menos 1 instrumento: {len(frame_diversity)}')

if not frame_diversity:
    print('AVISO: nenhum frame com instrumento no test split. Verificar conversao 2.3.')
else:
    # Naming convention da cell 2.3: <VID>_<frame_id>.png (e.g. VID11_002676.png)
    def _video_id(name: str) -> str | None:
        m = re.match(r'(VID\d+)', name)
        return m.group(1) if m else None

    def _frame_num(name: str) -> int:
        m = re.search(r'_(\d+)\.png$', name)
        return int(m.group(1)) if m else -1

    by_video: dict[str, list[str]] = defaultdict(list)
    for fname in frame_diversity:
        vid = _video_id(fname)
        if vid:
            by_video[vid].append(fname)

    if by_video:
        # Seleciona video com maior soma de diversidade de classes
        best_vid = max(
            by_video.items(),
            key=lambda kv: sum(frame_diversity.get(f, 0) for f in kv[1]),
        )
        vid_name, vid_frames = best_vid
        vid_frames = sorted(vid_frames, key=_frame_num)
        print(f'Video escolhido: {vid_name} ({len(vid_frames)} frames com instrumento)')
        sample = vid_frames[:30]
    else:
        # Sem agrupamento por video: usa frames mais ricos
        print('AVISO: regex de video nao casou. Usando 30 frames mais ricos em classes.')
        sample = sorted(frame_diversity, key=frame_diversity.get, reverse=True)[:30]

    print(f'Selecionados {len(sample)} frames para video anotado')

    # Rodar inferencia + anotar
    tmp_dir = Path(tempfile.mkdtemp(prefix='annot_ct20_'))
    try:
        for i, fname in enumerate(sample):
            img_path = test_img_dir / fname
            pred = best_model(str(img_path), conf=0.25, verbose=False)[0]
            annotated = pred.plot()  # numpy BGR com bboxes desenhadas
            cv2.imwrite(str(tmp_dir / f'frame_{i:04d}.png'), annotated)

        # Empacotar em MP4 via ffmpeg @ 8 fps
        mp4_out = DRIVE_RUNS / RUN_NAME / 'validation_instruments.mp4'
        mp4_out.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            [
                'ffmpeg', '-y',
                '-framerate', '8',
                '-i', str(tmp_dir / 'frame_%04d.png'),
                '-c:v', 'libx264',
                '-pix_fmt', 'yuv420p',
                str(mp4_out),
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        size_mb = mp4_out.stat().st_size / (1024 * 1024)
        print(f'\nVideo anotado salvo: {mp4_out}')
        print(f'Tamanho: {size_mb:.2f} MB ({len(sample)} frames @ 8 fps)')
        print('Usar esse MP4 como evidencia visual no relatorio tecnico.')
    finally:
        _shutil.rmtree(tmp_dir, ignore_errors=True)

## 5. Export

Confirma os artefatos sincronizados no Drive. Baixar `best.pt` pela barra lateral do Colab (ou pelo Drive), colocar em `models/yolov8n_surgical_ct20.pt` no repo e ajustar `YOLO_WEIGHTS_PATH` no `.env`.

In [ ]:
# Os artefatos ja foram sincronizados pelo callback do treino.
# Esta celula confirma o que esta no Drive em yolo_runs_cholectrack20/<RUN_NAME>/

drive_run = DRIVE_RUNS / RUN_NAME
print(f'Artefatos no Drive ({drive_run}):')
print()

key_files = [
    'weights/best.pt',
    'weights/last.pt',
    'results.png',
    'results.csv',
    f'../{RUN_NAME}_test/confusion_matrix.png',
    'val_batch0_pred.jpg',
    'validation_instruments.mp4',
]

for rel in key_files:
    f = drive_run / rel
    if f.exists():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'  OK   {rel:<45s} ({size_mb:.2f} MB)')
    else:
        print(f'  SKIP {rel:<45s} (nao gerado)')

## Fontes

- CholecTrack20: Nwoye et al. (2023), dataset de tracking de instrumentos cirurgicos, Synapse [syn53182642](https://www.synapse.org/Synapse:syn53182642/wiki/628404), codigo de referencia [CAMMA-public/cholectrack20](https://github.com/CAMMA-public/cholectrack20)
- CholecSeg8k (baseline): Hong et al. (2020), [arXiv:2012.12463](https://arxiv.org/abs/2012.12463) (CC BY-NC-SA 4.0)
- Ultralytics YOLOv8: [docs.ultralytics.com](https://docs.ultralytics.com/)
- Synapse Python client: [python-docs.synapse.org](https://python-docs.synapse.org/)
- Repo do projeto: [github.com/joalissonborges94/medica-ia-multimodal](https://github.com/joalissonborges94/medica-ia-multimodal)